# Day 1 - Lesson 5 실습: Gradio로 챗봇 웹 UI 만들기

> **이론 문서**: `Day1_Lesson5_Theory.md`  
> **선행 실습**: `Day1_Lesson1` ~ `Day1_Lesson4` 완료 (특히 4교시 메모리 체인)  
> **실습 시간**: 약 70분  
> **사용 모델**: GPT-4o-mini

---

## 🎯 이 노트북에서 만들 것

지금까지 만든 챗봇은 모두 **터미널/노트북 셀 안에서만** 동작했어요.  
오늘은 **브라우저에서 동작하는 진짜 웹 UI**로 발전시킵니다.

```
1️⃣ Interface — 함수 1개를 UI 1개로
       ↓
2️⃣ ChatInterface — 챗봇 UI 한 줄
       ↓
3️⃣ Blocks 입문 — 자유 레이아웃
       ↓
4️⃣ 핵심 컴포넌트 6가지 (Textbox, Dropdown, Slider, …)
       ↓
5️⃣ 레이아웃 (Row, Column, Tab)
       ↓
6️⃣ 이벤트 핸들러 (.click, .submit)
       ↓
7️⃣ LangChain 결합 — 패턴 1·2·3
       ↓
🎁 도전과제: 페르소나 + 메모리 + Blocks 완전체
```

## 📖 이론 매핑

| 이론 문서 섹션 | 이 노트북에서 다루는 코드 |
|---|---|
| 2부 ①. Interface | Step 1 |
| 2부 ②. ChatInterface | Step 2 |
| 2부 ③. Blocks | Step 3 |
| 3부. 핵심 컴포넌트 | Step 4 |
| 4부. 레이아웃 | Step 5 |
| 5부. 이벤트 핸들러 | Step 6 |
| 6부 패턴 1/2/3 + 4교시 결합 | Step 7 (단순/페르소나/메모리) |
| (6교시 예고 — 결과물) | 도전과제 1·2 |

---

## ⚠️ 시작 전 꼭 알아둘 3가지

1. **각 launch 셀은 서버를 띄웁니다.** 새 UI를 띄우기 전엔 `gr.close_all()`을 호출해 이전 서버를 정리합니다 (포트 충돌 방지).
2. **Colab에서 외부 공유**가 필요하면 `demo.launch(share=True)`로 72시간 유효한 공개 URL을 받을 수 있습니다.
3. **노트북 안에 iframe**으로 표시됩니다. 별도 브라우저 탭이 안 열려도 정상이에요.


---

# 🛠 Step 0. 환경 준비


In [1]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab 환경입니다.")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter 환경입니다.")


✅ Google Colab 환경입니다.


In [ ]:
# 2️⃣ Gradio + LangChain 패키지 설치
%pip install -q gradio langchain langchain-openai langchain-core python-dotenv


In [4]:
# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 로드 완료" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


✅ API 키 로드 완료


In [ ]:
# 4️⃣ 공통 LLM과 Gradio 임포트
from langchain_openai import ChatOpenAI
import gradio as gr

# Gradio 6 uses messages by default; earlier versions require type.
chatbot_options = {"type": "messages"} if int(gr.__version__.split(".")[0]) < 6 else {}

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
print(f"✅ Gradio 버전: {gr.__version__}")


---

# 1️⃣ Step 1. `gr.Interface` — 함수 하나를 UI 하나로

> **이론 매핑**: 2부 ①. Interface

가장 단순한 방식입니다. **함수 1개를 그대로 UI로 만들어주는** 도구예요.

### 동작 원리

```
[입력 컴포넌트] → [내 Python 함수] → [출력 컴포넌트]
```

함수 시그니처와 inputs/outputs만 정의하면 끝.


In [ ]:
# 가장 단순한 Interface — LLM 없이 그냥 인사 함수
def hello(name):
    return f"안녕하세요, {name}님! 반가워요 👋"

gr.close_all()  # 이전 서버 정리

demo = gr.Interface(
    fn=hello,                  # 호출할 함수
    #inputs="text",             # 입력: 텍스트박스 (단축 표기)
    inputs=gr.Textbox(label="이름", placeholder="홍길동"),
    outputs="text",            # 출력: 텍스트박스
    title="인사 봇",
    description="이름을 입력하면 인사해줍니다.",
)
demo.launch()


✅ **결과 확인**: 노트북 아래에 입력창 + Submit 버튼 + 출력창이 나타났나요?  
이름을 입력하고 Submit을 눌러보세요.

> 💡 위의 `inputs="text"`는 단축 표기예요. 더 세밀한 제어가 필요하면 `inputs=gr.Textbox(label="이름", placeholder="홍길동")`처럼 컴포넌트 객체를 직접 넘깁니다.


---

# 2️⃣ Step 2. `gr.ChatInterface` — 챗봇 UI 한 줄

> **이론 매핑**: 2부 ②. ChatInterface

ChatGPT 같은 챗봇 UI를 **한 줄**로 만들 수 있어요.

### `chat_fn`의 약속

```python
def chat_fn(message, history):
    # message: 사용자가 방금 보낸 새 메시지 (str)
    # history: Gradio가 화면용으로 관리하는 대화 내역
    return "AI 답변 (str)"
```

⚠️ **`history` 인자를 안 써도 함수 시그니처엔 반드시 있어야** 합니다. 없으면 에러나요.


In [9]:
# 메아리 챗봇 — LLM 없이 메시지를 그대로 돌려줌 (시그니처 익히기 용도)
def echo(message, history):
    return f"방금 '{message}'라고 하셨네요!"

gr.close_all()

demo = gr.ChatInterface(
    fn=echo,
    title="🦜 메아리 봇",
    description="입력한 메시지를 그대로 돌려주는 가장 단순한 챗봇",
)
demo.launch()


Closing server running on port: 7860


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c7f0184af5c6caa55c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ **결과 확인**: ChatGPT 같은 채팅 UI가 나타났나요? 메시지를 보내면 그대로 돌아옵니다.

> 💡 채팅 히스토리·말풍선·전송 버튼·"채팅 다시 시작" 버튼이 자동으로 만들어져요. 이 정도 UI를 직접 짜면 React로 하루 종일 걸립니다.


---

# 3️⃣ Step 3. `gr.Blocks` — 자유 레이아웃

> **이론 매핑**: 2부 ③. Blocks

Interface와 ChatInterface는 정해진 틀이 있어요.  
**여러 컴포넌트를 자유롭게 배치**하려면 `Blocks`를 씁니다. `with` 블록 구조가 특징이에요.

### 가장 단순한 Blocks


In [10]:
gr.close_all()

# with gr.Blocks() as demo: ← 이 안에 컴포넌트를 자유롭게 배치
with gr.Blocks() as demo:
    gr.Markdown("# 🎯 첫 Blocks 앱")           # 제목
    gr.Markdown("아래에 텍스트를 입력하면 그대로 보입니다.")

    input_box = gr.Textbox(label="입력")
    output_box = gr.Textbox(label="출력", interactive=False)
    btn = gr.Button("복사하기", variant="primary")

    # 버튼 클릭 시 input 값을 output에 그대로 전달
    btn.click(fn=lambda x: x, inputs=input_box, outputs=output_box)

demo.launch()


Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a88d8800f21b498ebc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ **결과 확인**: 제목, 입력, 출력, 버튼이 위에서 아래로 차곡차곡 쌓인 모습이 보이나요?

### 🔑 Blocks의 핵심 3가지

1. **`with gr.Blocks() as demo:`** — 이 안에 컴포넌트 정의
2. **컴포넌트 정의 = 변수에 저장** — `input_box = gr.Textbox(...)`
3. **이벤트 연결** — `btn.click(fn=..., inputs=..., outputs=...)`

이 3단 구조가 모든 Blocks 앱의 뼈대입니다.


---

# 4️⃣ Step 4. 주요 컴포넌트 6가지

> **이론 매핑**: 3부. 핵심 컴포넌트

LLM 챗봇에서 자주 쓰는 컴포넌트 6가지를 한 화면에서 갤러리처럼 봅시다.


In [11]:
gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🧱 컴포넌트 갤러리")

    # 1. Textbox — 가장 많이 쓰는 텍스트 입력
    txt = gr.Textbox(label="질문 입력", placeholder="여기에 입력하세요...", lines=2)

    # 2. Dropdown — 페르소나·모델 선택에 사용
    persona = gr.Dropdown(
        choices=["분석가", "마케터", "친구"],
        value="친구",
        label="페르소나 선택",
    )

    # 3. Slider — temperature, max_tokens 같은 수치 조절
    temperature = gr.Slider(minimum=0, maximum=2, step=0.1, value=0.7, label="Temperature")

    # 4. Radio — 3-4개 옵션 중 하나
    style = gr.Radio(choices=["짧게", "보통", "자세히"], value="보통", label="답변 스타일")

    # 5. Checkbox — True/False
    use_stream = gr.Checkbox(value=True, label="스트리밍 사용")

    # 6. Button — 클릭 트리거 (variant로 색상 지정)
    btn = gr.Button("전송", variant="primary")  # primary=파랑, secondary=회색, stop=빨강

    # 출력 — 입력 값들을 모아서 보여주기
    out = gr.Textbox(label="현재 설정", interactive=False, lines=5)

    def show_settings(t, p, temp, s, u):
        return f"질문: {t}\n페르소나: {p}\nTemp: {temp}\n스타일: {s}\n스트림: {u}"

    btn.click(
        fn=show_settings,
        inputs=[txt, persona, temperature, style, use_stream],  # 입력 순서 = 함수 인자 순서
        outputs=out,
    )

demo.launch()


Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://274c290dc5228a4e55.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ **결과 확인**: 6가지 컴포넌트를 모두 만져보고 "전송"을 누르면, 입력한 값들이 한꺼번에 출력됩니다.

### 컴포넌트 한눈 표

| 컴포넌트 | 주요 용도 |
|---|---|
| `Textbox` | 텍스트 입력/출력 (가장 많이 씀) |
| `Dropdown` | 페르소나·모델 선택 |
| `Slider` | temperature, max_tokens |
| `Radio` | 3-4개 옵션 중 하나 |
| `Checkbox` | True/False 토글 |
| `Button` | 클릭 트리거 (`variant`: primary/secondary/stop) |

> 💡 핵심: 컴포넌트는 **변수에 저장**해두고, 이벤트 핸들러의 `inputs`·`outputs`에서 그 변수를 가리킵니다.


---

### 🐛 잠깐, 이상한 코드 하나 — *조용한* 버그 디버깅

다음 코드는 *에러는 안 납니다*. 그런데 버튼을 누르면 *결과가 이상하게* 나옵니다.  
**AI에게 물어보기 전에**, 코드를 5분만 노려보세요. 어디가 문제일까요?

> 💡 Gradio 입문자가 가장 자주 만나는 함정 1순위입니다.

In [12]:
# 🐛 이름과 나이를 입력하면 인사를 돌려주는 함수 — 그런데 결과가 이상함
def show_user(name, age):
    return f"{name}님은 {age}살입니다."

gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🐛 무엇이 잘못됐을까?")
    name_box = gr.Textbox(label="이름")
    age_box = gr.Number(label="나이", value=30)
    out = gr.Textbox(label="결과", interactive=False)
    btn = gr.Button("확인", variant="primary")

    # ↓ 이 한 줄에 버그가 있음
    btn.click(show_user, inputs=[age_box, name_box], outputs=out)

demo.launch()

Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b9d1c03059cdc19e45.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🤔 **관찰 포인트**:
- 이름에 "진환", 나이에 30을 넣고 *확인* 버튼을 눌러보세요.
- 결과가 어떻게 나왔나요? "*30님은 진환살입니다*" 같은 게 나오지 않았나요?
- 에러가 안 났는데도 *결과가 이상한* 이유는 무엇일까요?

> 💡 **교훈**: Gradio의 `inputs=[...]` 리스트는 **함수 매개변수 순서와 정확히 맞춰야** 합니다.  
> 함수가 `show_user(name, age)`이면 `inputs`도 `[name_box, age_box]` 순서여야 해요.
>
> **버그 시리즈 진행 상황**:
> - 1교시 분류기: *조용한* 버그 (temperature 잘못)
> - 2교시 템플릿: *시끄러운* `KeyError` (변수 이름 불일치)
> - 4교시 메모리: *시끄러운* `ValueError` (config 누락)
> - 오늘 Gradio: 다시 *조용한* 버그 (inputs 순서 어긋남)
>
> "에러가 없다 = 정상" 이 절대 아닙니다. 결과가 *기대대로 나왔는가*까지 확인하는 습관이 필수예요.

#### ✏️ 직접 고쳐보기

위 셀을 *복사해서* 아래에 새 셀을 만들고 `inputs=` 순서를 바로잡아 동작하게 고쳐보세요.

---

# 5️⃣ Step 5. 레이아웃 — `Row`, `Column`, `Tab`

> **이론 매핑**: 4부. 레이아웃

기본은 **세로로 쌓이는** 구조예요. 가로로 배치하거나 탭으로 분리하려면 레이아웃 컨테이너를 씁니다.

### Row + Column 좌우 분할 (실무 가장 흔한 패턴)


In [13]:
gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🪟 좌우 분할 레이아웃")

    with gr.Row():                              # ← Row 안의 것은 가로로 배치
        with gr.Column():                       # ← Column 안의 것은 세로로 배치
            gr.Markdown("### 왼쪽 (입력)")
            left_input = gr.Textbox(label="질문")
            left_btn = gr.Button("전송", variant="primary")

        with gr.Column():
            gr.Markdown("### 오른쪽 (출력)")
            right_output = gr.Textbox(label="답변", lines=5, interactive=False)

    left_btn.click(fn=lambda x: f"답변: {x}", inputs=left_input, outputs=right_output)

demo.launch()


Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1285464d096f26f20c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ **결과 확인**: 화면이 좌우로 깔끔하게 갈렸나요? **왼쪽엔 입력, 오른쪽엔 출력**이 정석 구조입니다.

### Tab으로 화면 분리


In [14]:
gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 📑 탭으로 분리된 앱")

    with gr.Tab("💬 채팅"):
        gr.Markdown("여기에 채팅 UI를 배치")
        gr.Textbox(label="메시지")

    with gr.Tab("⚙️ 설정"):
        gr.Slider(0, 2, value=0.7, label="Temperature")
        gr.Dropdown(["gpt-4o-mini", "gpt-4o"], value="gpt-4o-mini", label="모델")

    with gr.Tab("📊 사용량"):
        gr.Markdown("토큰 사용량 차트가 들어갈 자리")

demo.launch()


Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://298d1fa3742101b757.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ **결과 확인**: 상단에 탭 3개가 보이고, 클릭하면 다른 화면이 보이나요?  
> 💡 5일 코스 최종 결과물에서 "챗봇 / 설정 / 로그" 같은 식으로 화면을 나눌 때 씁니다.


---

# 6️⃣ Step 6. 이벤트 핸들러 — `.click`, `.submit`

> **이론 매핑**: 5부. 이벤트 핸들러

| 이벤트 | 트리거 | 자주 쓰는 곳 |
|---|---|---|
| `.click` | 버튼 클릭 | 일반 버튼 |
| `.submit` | 텍스트박스에서 엔터 | 메시지 전송 |
| `.change` | 값이 바뀔 때 | 드롭다운 변경 즉시 반응 |

### 같은 함수를 click과 submit 둘 다에 연결하기 (실무 흔함)

사용자가 **버튼을 누르든 엔터를 치든** 같은 동작이 일어나게 합니다.


In [15]:
gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# ⚡ 클릭 + 엔터 동시 지원")

    input_box = gr.Textbox(label="질문", placeholder="입력 후 엔터 또는 전송 버튼")
    submit_btn = gr.Button("전송", variant="primary")
    output_box = gr.Textbox(label="답변", interactive=False)

    # 처리 함수 1개
    def process(question):
        return f"질문 '{question}'에 대한 답변입니다."

    # 버튼 클릭 → process 실행
    submit_btn.click(fn=process, inputs=input_box, outputs=output_box)
    # 엔터 → process 실행 (같은 함수!)
    input_box.submit(fn=process, inputs=input_box, outputs=output_box)

demo.launch()


Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f3a6e279067b5011c4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ **결과 확인**: 엔터를 쳤을 때와 버튼을 눌렀을 때 **둘 다 동일하게 동작**하나요?

> 💡 챗봇에서 **거의 필수 패턴**입니다. 사용자가 엔터로 메시지 보내는 걸 더 선호하거든요.


---

# 7️⃣ Step 7. LangChain과 결합 — 3가지 패턴

> **이론 매핑**: 6부. LangChain과 결합 패턴

지금까지 배운 Gradio에 LLM을 끼워봅시다. **3단계로 점점 발전**시킵니다.

### 🔹 패턴 1. Interface + 단순 LLM 호출


In [16]:
# 가장 단순 — 함수 안에서 LLM 한 번 호출
def ask_llm(question: str) -> str:
    return llm.invoke(question).content

gr.close_all()

demo = gr.Interface(
    fn=ask_llm,
    inputs=gr.Textbox(label="질문", placeholder="LLM에게 물어볼 질문을 입력하세요"),
    outputs=gr.Textbox(label="답변", lines=8),
    title="🤖 단순 LLM 챗봇",
    description="Interface + ChatOpenAI 최소 결합",
)
demo.launch()


Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://03cd358b528d19bd5f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ **결과 확인**: 진짜로 LLM이 답변하는 챗봇이 1분 만에 완성됐어요. **이게 Gradio의 위력입니다.**


### 🔹 패턴 2. Blocks + 페르소나 드롭다운

이론 6부 패턴 2와 동일. **드롭다운으로 페르소나 변경**, system 메시지가 자동으로 바뀝니다.


In [17]:
from langchain_core.prompts import ChatPromptTemplate

# 페르소나 사전 (system 메시지)
PERSONAS = {
    "꼼꼼한 분석가": "당신은 데이터를 꼼꼼히 분석하는 시니어 데이터 분석가입니다. 수치와 근거를 중시합니다.",
    "톡톡 튀는 마케터": "당신은 트렌드를 빠르게 읽는 마케터입니다. 활기차고 창의적인 톤으로 답하세요.",
    "친한 친구": "당신은 사용자의 절친한 친구입니다. 반말로 편하게 답하세요.",
    # 🎯 미션: 본인만의 4번째 페르소나를 추가하세요.
    #         (예시: "꼬치꼬치 캐묻는 시니어 리뷰어", "츤데레 데이터 멘토" 등)
    #         조건: system 메시지를 *2문장 이상*으로 구체화할 것
    "____": "____",
}

# 프롬프트 + 체인 (2교시 패턴)
persona_prompt = ChatPromptTemplate.from_messages([
    ("system", "{persona_msg}"),
    ("human", "{question}"),
])
persona_chain = persona_prompt | llm


def chat_with_persona(persona_name: str, question: str) -> str:
    """드롭다운에서 선택된 페르소나로 LLM 호출"""
    result = persona_chain.invoke({
        "persona_msg": PERSONAS[persona_name],
        "question": question,
    })
    return result.content

In [18]:
gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🎭 페르소나 챗봇 v1 (단발 질문)")

    with gr.Row():
        with gr.Column():
            persona_dd = gr.Dropdown(
                choices=list(PERSONAS.keys()),
                value="친한 친구",
                label="페르소나 선택",
            )
            question_box = gr.Textbox(label="질문", lines=3)
            submit_btn = gr.Button("전송", variant="primary")

        with gr.Column():
            answer_box = gr.Textbox(label="답변", lines=10, interactive=False)

    # 클릭과 엔터 둘 다 지원
    submit_btn.click(chat_with_persona, [persona_dd, question_box], answer_box)
    question_box.submit(chat_with_persona, [persona_dd, question_box], answer_box)

demo.launch()


Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0d258354d18bb98013.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ **결과 확인**: 페르소나를 바꿔가며 같은 질문(예: "AI가 뭐예요?")을 던져보세요. 답변 톤이 완전히 달라집니다.

### 🎲 예측 게임 — 이 챗봇은 *기억할* 수 있을까?

방금 만든 패턴 2 챗봇에 다음 메시지를 *연달아* 보내봅시다:
1. `내 이름은 진환이야`
2. `내 이름이 뭐였지?`

예측해보세요:

| 결과 | 같은 페르소나로 연달아 | 페르소나 바꾸고 연달아 |
|---|:---:|:---:|
| 챗봇이 "진환"을 기억함 | ☐ 예 / ☐ 아니오 | ☐ 예 / ☐ 아니오 |
| 페르소나 톤은 *일관됨* | ☐ 예 / ☐ 아니오 | ☐ 예 / ☐ 아니오 |

체크한 뒤 실제로 챗봇 UI에서 두 메시지를 보내 확인하세요.  
*왜 그런 결과가 나오는지* 코드를 다시 보면서 생각해보세요.

> 💡 **힌트**: `persona_prompt`에 `MessagesPlaceholder`가 있나요? 4교시에서 메모리 만들 때 무엇이 필요했나요?
>
> ⚠️ 답을 확인했다면: 이 챗봇은 **대화를 기억하지 못합니다** (메모리 없음).  
> 다음 패턴 3에서 메모리를 결합해서 *진짜 챗봇*을 만듭니다.

### 🔹 패턴 3. ChatInterface + 4교시 메모리 체인 ⭐

이론 6부 패턴 3과 동일. **이게 오늘의 메인 결과물**이에요.


In [19]:
# 4교시에 만든 메모리 체인을 그대로 가져와서 재구성
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1) MessagesPlaceholder가 포함된 프롬프트
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

# 2) 세션 저장소
store = {}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 3) 체인을 메모리로 감싸기
chain_with_memory = RunnableWithMessageHistory(
    memory_prompt | llm,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### 🤔 실행 전에 1분 멈춤

다음 셀에서 `chat_fn(message, history)`가 정의됩니다.  
`history`는 Gradio가 자동으로 넘겨주는 *대화 화면용* 데이터예요.

질문: 우리가 직접 만든 **LangChain의 `store`**와 Gradio의 **`history`**는 같은 것일까요, 다른 것일까요?  
실행 *전에* 예측해보세요:

- `history`의 *타입*은? **list? dict? 다른 객체?**
- 그 안에 든 *개별 항목*은 어떤 형태? (튜플? 딕셔너리? 메시지 객체?)
- LangChain의 `store["default"]`와 *내용이 같을까* 다를까?

> 💡 셀을 실행하고 챗봇에 메시지를 한두 번 보내면 노트북 출력에 `history`의 정체가 찍힙니다.

In [20]:
# Gradio ChatInterface와 결합
def chat_fn(message, history):
    """ChatInterface가 요구하는 시그니처 (message, history)"""
    # 🔍 예측 확인: Gradio가 넘겨주는 history는 어떤 모양?
    print(f"📦 history의 타입: {type(history).__name__}, 길이: {len(history)}", flush=True)
    if history:
        print(f"   첫 항목: {history[0]}", flush=True)

    # history는 Gradio가 화면용으로 자동 관리하므로 우리는 안 씀
    # LangChain 메모리는 store가 별도로 관리
    result = chain_with_memory.invoke(
        {"question": message},
        config={"configurable": {"session_id": "default"}},
    )

    # 🔍 비교 확인: LangChain의 store는 어떤 모양?
    print(f"📦 store의 타입: {type(store).__name__}, 세션 수: {len(store)}", flush=True)
    print(f"   store['default'] 메시지 수: {len(store['default'].messages)}", flush=True)
    print("-" * 50, flush=True)

    return result.content


gr.close_all()

demo = gr.ChatInterface(
    fn=chat_fn,
    title="🤖 메모리 챗봇",
    description="4교시에서 만든 메모리 체인 + Gradio ChatInterface",
)
demo.launch()

Closing server running on port: 7860


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ec62c582e8fb6ef0e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 🧪 테스트 시나리오

채팅창에 차례대로 입력해보세요:

1. `내 이름은 진환이야`
2. `취미는 모델 비행기 만들기야`  
3. `내 이름이 뭐였지?`  ← **"진환"이 나와야 정상**
4. `내 취미가 뭐였지?` ← **"모델 비행기"가 나와야 정상**

✅ **결과 확인**: 챗봇이 이전 대화를 기억하나요?

### 🔍 history vs store — 예측 확인

위 시나리오를 진행하면서 노트북 출력에 찍힌 `history`와 `store`를 비교해보세요:

- Gradio의 `history`는 *비어있는 채로* 함수에 들어왔나요, 아니면 *이전 대화가 누적*되어 있었나요?
- LangChain의 `store["default"].messages`는 호출이 거듭될수록 *늘어나는가* 같은가요?
- 둘은 *같은 데이터*인가요, 아니면 *다른 곳*에서 관리되는 두 사본인가요?

> 🎯 **핵심**: 4교시에 만든 `chain_with_memory`를 **한 줄도 안 바꾸고** 그대로 가져왔어요.  
> Gradio의 `chat_fn` 안에서 `chain_with_memory.invoke(...)`만 호출하면 끝.  
> **LangChain과 Gradio의 깔끔한 분리** — 이게 우리 코스의 설계 철학입니다.
>
> Gradio의 `history`는 *화면 표시용*, LangChain의 `store`는 *LLM이 참고하는 실제 메모리*.  
> 둘은 별개로 동작하는 **두 개의 저장소**입니다.

---

# 📝 정리: 이번 실습에서 만든 것

- [x] `gr.Interface`로 함수 → UI 변환
- [x] `gr.ChatInterface`로 챗봇 UI 한 줄
- [x] `gr.Blocks`로 자유 레이아웃
- [x] 6가지 컴포넌트 + Row/Column/Tab 레이아웃
- [x] `.click`과 `.submit` 이벤트 핸들러
- [x] LangChain 결합 패턴 1·2·3 (Interface · Blocks · ChatInterface+메모리)

### 🔑 외워둘 패턴

```python
# Blocks 골격
with gr.Blocks() as demo:
    gr.Markdown("# 제목")
    
    with gr.Row():
        with gr.Column():
            input_box = gr.Textbox(label="입력")
            btn = gr.Button("전송", variant="primary")
        with gr.Column():
            output_box = gr.Textbox(label="출력")
    
    btn.click(my_function, inputs=input_box, outputs=output_box)

gr.close_all()
demo.launch()
```

### ➡️ 다음 시간 예고

6교시에는 **응답 스트리밍**과 **7교시 자율 실습 결과물 명세**를 다룹니다.

```python
# 일반 호출 (5초 후 답변 한 번에 나옴)
result = chain.invoke({"question": "..."})

# 스트리밍 (글자 한 개씩 점진적으로 나옴 = ChatGPT 같은 효과)
for chunk in chain.stream({"question": "..."}):
    yield chunk.content
```

오늘 만든 챗봇에 스트리밍을 더하면 사용자 체감 속도가 크게 좋아집니다.

---

이제 도전과제로 손에 익혀봅시다 🎯


---

# 🎁 도전과제

## 도전과제 1. 페르소나 + 메모리 + Blocks 완전체

지금까지 배운 걸 모두 합칩니다.  
**페르소나 드롭다운 + 채팅 히스토리 + 자동 메모리**가 모두 있는 챗봇이에요.

### 명세

```
┌──────────────────────────────────┐
│  🎭 페르소나 챗봇                  │
├──────────────────────────────────┤
│  [페르소나: 분석가 ▾]              │
│                                  │
│  💬 대화창 (gr.Chatbot)            │
│  👤 안녕                          │
│  🤖 안녕하세요!                    │
│                                  │
│  [메시지 입력...]      [전송]      │
└──────────────────────────────────┘
```

### 힌트

- `gr.Chatbot` 컴포넌트로 대화 표시 (`type="messages"` 옵션 사용)
- 드롭다운 값을 system 메시지에 넣기 위해 페르소나별 프롬프트 체인 필요
- `chain_with_memory.invoke({"question": ...}, config={...})`로 호출
- 한 번 호출하고 나면 history 리스트에 (사용자, AI) 한 쌍 추가


In [ ]:
# 📌 페르소나 + 메모리 + Blocks — 빈칸을 채우세요

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

PERSONAS_FULL = {
    "꼼꼼한 분석가": "당신은 데이터를 꼼꼼히 분석하는 시니어 분석가입니다. 수치와 근거를 중시합니다.",
    "톡톡 튀는 마케터": "당신은 트렌드 마케터입니다. 활기차고 창의적으로 답하세요.",
    "친한 친구": "당신은 사용자의 절친한 친구입니다. 반말로 편하게 답하세요.",
}

# 👇 빈칸 1: system에 {persona}, history 자리, human에 {question}
full_prompt = ChatPromptTemplate.from_messages([
    ("system", "{persona}"),  # ← 빈칸
    MessagesPlaceholder("history"),                  # ← 빈칸 (MessagesPlaceholder)
    ("human", "{question}"),
])

full_store = {}
def get_full_history(session_id):
    if session_id not in full_store:
        full_store[session_id] = InMemoryChatMessageHistory()
    return full_store[session_id]

full_chain = RunnableWithMessageHistory(
    full_prompt | llm,
    get_full_history,
    input_messages_key="question",
    history_messages_key="history",
)


def respond(message, history, persona_name):
    """
    Gradio에서 호출되는 함수
    - message: 사용자가 방금 보낸 새 메시지
    - history: gr.Chatbot이 화면용으로 관리하는 리스트
    - persona_name: 드롭다운에서 선택된 페르소나 이름
    """
    # 👇 빈칸 2: 메모리 체인 호출 (페르소나 메시지 + 질문, session_id="default")
    result = full_chain.invoke(
        {"persona": PERSONAS_FULL[persona_name], "question": message},
        config={"configurable": {"session_id": "default"}},  # ← 빈칸
    )
    answer = result.content

    # gr.Chatbot의 messages 형식: [{"role": "user", ...}, {"role": "assistant", ...}]
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]
    return "", history  # 첫 반환값은 입력창 비우기 용도


gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🎭 페르소나 챗봇 (메모리 포함)")

    persona_dd = gr.Dropdown(
        choices=list(PERSONAS_FULL.keys()),
        value="친한 친구",
        label="페르소나",
    )

    chatbot = gr.Chatbot(label="대화창", height=400, **chatbot_options)
    msg_box = gr.Textbox(label="메시지", placeholder="입력 후 엔터")
    send_btn = gr.Button("전송", variant="primary")

    # 클릭과 엔터 모두 같은 함수로 연결
    send_btn.click(respond, [msg_box, chatbot, persona_dd], [msg_box, chatbot])
    msg_box.submit(respond, [msg_box, chatbot, persona_dd], [msg_box, chatbot])

demo.launch()


✅ **결과 확인**:
1. 페르소나를 "친한 친구"로 두고 "내 이름은 진환이야" 입력
2. "내 이름이 뭐였지?" 물어보면 진환이라고 기억해야 함
3. 페르소나를 "꼼꼼한 분석가"로 바꾸고 다시 질문 → 톤이 달라지는지 확인

---

## 도전과제 2. 대화 초기화 — 3가지 전략 비교 ⭐

위 챗봇에 **"🗑 대화 초기화"** 버튼을 추가하려고 합니다.  
하지만 *어떻게 초기화할 것인가*에 3가지 선택지가 있어요. **셋 다 만들어보고 어느 게 옳은가 판단**하세요.

| 전략 | 화면(`chatbot`) | 메모리(`full_store`) |
|---|:---:|:---:|
| A | ✅ 비움 | ❌ 그대로 |
| B | ❌ 그대로 | ✅ 비움 |
| C ⭐ | ✅ 비움 | ✅ 비움 |

각 전략을 실제로 만들어보고, 같은 시나리오로 테스트해서 *어디서 문제가 생기는지* 직접 봐야 합니다.

In [ ]:
# 📌 3가지 초기화 전략 — 셋 다 만들어서 비교

# 전략 A: 화면만 비움 (메모리는 그대로)
def reset_screen_only():
    """⚠️ 화면만 비우고 메모리는 그대로 둠"""
    return []  # 화면(chatbot)에 빈 리스트만 반환

# 전략 B: 메모리만 비움 (화면은 그대로)
def reset_memory_only():
    """⚠️ 메모리만 비우고 화면은 그대로 둠"""
    if "default" in full_store:
        full_store["default"].clear()
    # 반환값 없음 → 화면 출력 변화 없음
    return gr.update()  # 화면을 *건드리지 않음*

# 전략 C: 둘 다 비움 ⭐
def reset_both():
    """✅ 화면도 비우고 메모리도 비움"""
    if "default" in full_store:
        full_store["default"].clear()
    return []


gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🎭 페르소나 챗봇 — 3가지 초기화 전략 비교")

    persona_dd = gr.Dropdown(
        choices=list(PERSONAS_FULL.keys()),
        value="친한 친구",
        label="페르소나",
    )

    chatbot = gr.Chatbot(label="대화창", height=400, **chatbot_options)
    msg_box = gr.Textbox(label="메시지", placeholder="입력 후 엔터")

    with gr.Row():
        send_btn = gr.Button("전송", variant="primary")
        reset_a_btn = gr.Button("🅰️ 화면만 비우기", variant="secondary")
        reset_b_btn = gr.Button("🅱️ 메모리만 비우기", variant="secondary")
        reset_c_btn = gr.Button("🆑 둘 다 비우기", variant="stop")

    # 전송
    send_btn.click(respond, [msg_box, chatbot, persona_dd], [msg_box, chatbot])
    msg_box.submit(respond, [msg_box, chatbot, persona_dd], [msg_box, chatbot])

    # 3가지 초기화 전략
    reset_a_btn.click(fn=reset_screen_only, inputs=None, outputs=chatbot)
    reset_b_btn.click(fn=reset_memory_only, inputs=None, outputs=chatbot)
    reset_c_btn.click(fn=reset_both, inputs=None, outputs=chatbot)

demo.launch()

### 🧪 판단 시나리오

다음 절차를 *세 가지 초기화 버튼* 각각에 대해 반복해보세요:

1. 자기 이름을 알려준다 (예: "내 이름은 진환이야")
2. 챗봇이 이름을 기억하는지 확인 ("내 이름이 뭐였지?" → "진환")
3. 🅰️ / 🅱️ / 🆑 중 하나를 누른다
4. 다시 "내 이름이 뭐였지?" 물어본다
5. **화면 상태**와 **챗봇의 답변**을 모두 관찰

### 🤔 판단해보기

세 결과를 비교한 뒤 2-3줄로 답하세요:

1. 🅰️ (화면만 비움) 누르고 "내 이름이 뭐였지?" 물으면 어떻게 됐나요? *화면은 깨끗한데 챗봇은 뭘 답하는가?*
2. 🅱️ (메모리만 비움) 누르고 같은 질문을 하면? *화면엔 이전 대화가 남아있는데 챗봇은?*
3. 🆑 (둘 다 비움)만 *정상 동작*하는 이유는 무엇인가요?

> 💡 **이 연습의 진짜 의미**: UI(화면)와 *서버 상태*(LangChain store)는 **별개의 저장소**입니다.  
> 한쪽만 비우면 *사용자가 보는 것*과 *챗봇이 아는 것*이 어긋나는 **상태 불일치 버그**가 생겨요.  
> 실서비스에서 흔히 발생하는 사고 유형 — "표시는 안 되는데 데이터는 남아있음" 또는 그 반대 — 의 원형입니다.

---

## 🏁 모두 풀었다면

축하합니다! Day 1 Lesson 5 완료 🎉  
6교시에 **응답 스트리밍**까지 더하면 7교시 자율 실습에서 진짜 멋진 챗봇을 완성합니다.